# 06 -- Evaluation and Comparison

**Purpose:** Evaluate all trained models on the held-out test set. Produces:
- Accuracy, Precision, Recall, F1, PR-AUC (primary), ROC-AUC with bootstrap CIs
- Confusion matrices, PR curves, ROC curves
- Severity-stratified recall
- Swimming pool FP analysis with Clopper-Pearson CIs
- McNemar's test for pairwise model comparison
- Predictions CSV for downstream analysis (GradCAM, calibration, etc.)

**Compute:** Colab T4 GPU for inference; CPU for metrics and plots.

**Estimated runtime:** ~5 minutes per model.

**Script:** `scripts/evaluate.py`

**Note:** Replace `TIMESTAMP` placeholders with actual model filenames.
Run each model variant, then compare using `--compare_predictions_csv`.

In [ ]:
# Mount Google Drive and verify GPU runtime
from google.colab import drive
drive.mount('/content/drive')

import subprocess
gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if 'T4' in gpu_info.stdout:
    print('Runtime: Colab T4 GPU')
elif 'failed' in gpu_info.stderr.lower() or gpu_info.returncode != 0:
    print('Runtime: CPU only -- switch to a GPU runtime!')
else:
    print(f'Runtime: GPU detected\n{gpu_info.stdout[:200]}')

In [ ]:
%cd /content/drive/MyDrive/imagevalidation2

In [ ]:
# --- Baseline EfficientNetB0 ---
!python scripts/evaluate.py \
    --arch efficientnet \
    --model_path /content/drive/MyDrive/models/efficientnet_baseline_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results

In [ ]:
# --- Baseline ResNet50 ---
!python scripts/evaluate.py \
    --arch resnet50 \
    --model_path /content/drive/MyDrive/models/resnet50_baseline_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results

In [ ]:
# --- EfficientNetB0 + HNM (percentile) ---
!python scripts/evaluate.py \
    --arch efficientnet \
    --model_path /content/drive/MyDrive/models/efficientnet_hnm_percentile_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results

In [ ]:
# --- ResNet50 + HNM (percentile) ---
!python scripts/evaluate.py \
    --arch resnet50 \
    --model_path /content/drive/MyDrive/models/resnet50_hnm_percentile_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results

In [ ]:
# --- EfficientNetB0 + HNM (sweep) ---
!python scripts/evaluate.py \
    --arch efficientnet \
    --model_path /content/drive/MyDrive/models/efficientnet_hnm_sweep_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results

In [ ]:
# --- ResNet50 + HNM (sweep) ---
!python scripts/evaluate.py \
    --arch resnet50 \
    --model_path /content/drive/MyDrive/models/resnet50_hnm_sweep_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results

In [ ]:
# --- EfficientNetB0 Extended Training (no HNM -- control) ---
!python scripts/evaluate.py \
    --arch efficientnet \
    --model_path /content/drive/MyDrive/models/efficientnet_extended_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results

In [ ]:
# --- ResNet50 Extended Training (no HNM -- control) ---
!python scripts/evaluate.py \
    --arch resnet50 \
    --model_path /content/drive/MyDrive/models/resnet50_extended_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results

In [ ]:
# --- McNemar's test: EfficientNetB0 baseline vs HNM ---
!python scripts/evaluate.py \
    --arch efficientnet \
    --model_path /content/drive/MyDrive/models/efficientnet_hnm_percentile_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results \
    --compare_predictions_csv ./results/predictions/efficientnet_baseline_TIMESTAMP_predictions.csv

In [ ]:
# --- McNemar's test: EfficientNetB0 HNM vs ResNet50 baseline ---
!python scripts/evaluate.py \
    --arch efficientnet \
    --model_path /content/drive/MyDrive/models/efficientnet_hnm_percentile_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results \
    --compare_predictions_csv ./results/predictions/resnet50_baseline_TIMESTAMP_predictions.csv

In [ ]:
# Display all saved figures
import os
from IPython.display import Image, display

fig_dir = './results/figures'
if os.path.exists(fig_dir):
    for fname in sorted(os.listdir(fig_dir)):
        if fname.endswith('.png'):
            print(f'\n--- {fname} ---')
            display(Image(filename=os.path.join(fig_dir, fname), width=600))
else:
    print(f'Figures directory not found: {fig_dir}')